# Building Crack Detection and Localization using Deep Learning
**7th Semester Deep Learning Project, Superior University (Gold Campus)**

The system looks at a photo of a wall, floor or concrete surface and:
1. **Detects** if there is a crack (classification)
2. **Locates** the crack pixel by pixel and highlights it in red (segmentation)
3. **Reports** crack area and a rough severity level

| Part | Task | Dataset | Models |
|---|---|---|---|
| A | Crack / No crack | Concrete Crack Images (40,000 images) | Custom CNN (baseline) vs EfficientNetB0 |
| B | Crack location | Concrete crack segmentation (800 images + masks) | U-Net with MobileNetV2 encoder |
| C | Full system | Both models together | Crack Inspector |

**How to run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all` (about 25 to 35 minutes).

## 0. Setup

In [ ]:
!nvidia-smi -L
!pip install -q huggingface_hub

In [ ]:
import os, json, glob, time, zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from huggingface_hub import hf_hub_download, snapshot_download
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

QUICK = os.environ.get("QUICK_TEST") == "1"   # only for a fast code check, keep False for real training
SEED, AUTOTUNE = 42, tf.data.AUTOTUNE
CLS_IMG, SEG_IMG, BATCH = 224, 256, 32
CLASS_NAMES = ["No crack", "Crack"]
tf.random.set_seed(SEED); np.random.seed(SEED)
for d in ["data", "models", "results"]: os.makedirs(d, exist_ok=True)

# Part A: Crack Detection (Classification)
## A1. Download dataset
Concrete Crack Images for Classification (Özgenel, 2018): 40,000 images of 227x227 px, 20,000 with cracks and 20,000 without.
Already split: 24,000 train, 8,000 validation, 8,000 test.

In [ ]:
REPO = "mohammadnajeeb/concrete_crack_images"
for split in ["train", "validation", "test"]:
    if not os.path.exists(f"data/cls/{split}"):
        z = hf_hub_download(REPO, f"data/{split}.zip", repo_type="dataset")
        zipfile.ZipFile(z).extractall("data/cls")
for split in ["train", "validation", "test"]:
    for c in ["Negative", "Positive"]:
        print(f"{split:10s} {c:8s} {len(os.listdir(f'data/cls/{split}/{c}')):6d} images")

## A2. Load data

In [ ]:
def load_split(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        f"data/cls/{split}", class_names=["Negative", "Positive"], label_mode="binary",
        image_size=(CLS_IMG, CLS_IMG), batch_size=BATCH, shuffle=shuffle, seed=SEED)

train_ds = load_split("train", True)
val_ds   = load_split("validation", False)
test_ds  = load_split("test", False)
if QUICK:
    train_ds, val_ds, test_ds = train_ds.take(4), val_ds.take(2), test_ds.take(2)
train_ds, val_ds, test_ds = [d.prefetch(AUTOTUNE) for d in (train_ds, val_ds, test_ds)]

xb, yb = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1); plt.imshow(xb[i].numpy().astype("uint8")); plt.axis("off")
    plt.title(CLASS_NAMES[int(yb[i][0])], color="red" if yb[i][0] else "green")
plt.suptitle("Sample training images"); plt.tight_layout()
plt.savefig("results/cls_samples.png", dpi=110); plt.show()

## A3. Helpers

In [ ]:
def make_augment():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.1),
        layers.RandomContrast(0.15),
        layers.RandomBrightness(0.15, value_range=(0, 255)),
    ], name="augmentation")

def plot_history(h, title, fname, metric="accuracy"):
    plt.figure(figsize=(12, 4))
    for i, m in enumerate([metric, "loss"]):
        plt.subplot(1, 2, i + 1)
        plt.plot(h[m], label="train"); plt.plot(h["val_" + m], label="validation")
        plt.title(f"{title} - {m}"); plt.xlabel("epoch"); plt.legend()
    plt.tight_layout(); plt.savefig(fname, dpi=110); plt.show()

def evaluate_classifier(model, ds, name):
    y_true, y_prob = [], []
    for x, y in ds:
        y_prob.extend(model.predict(x, verbose=0).ravel()); y_true.extend(y.numpy().ravel())
    y_true, y_prob = np.array(y_true).astype(int), np.array(y_prob)
    y_pred = (y_prob > 0.5).astype(int)
    acc = float((y_pred == y_true).mean())
    print(f"{name} test accuracy: {acc:.4f}")
    return y_true, y_prob, y_pred, acc

## A4. Baseline: Custom CNN (built from scratch)

In [ ]:
def build_cnn():
    inp = layers.Input((CLS_IMG, CLS_IMG, 3))
    x = layers.Resizing(128, 128)(inp)
    x = make_augment()(x)
    x = layers.Rescaling(1. / 255)(x)
    for f in [32, 64, 128]:
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inp, out, name="custom_cnn")

cnn = build_cnn()
cnn.compile(tf.keras.optimizers.Adam(1e-3), "binary_crossentropy", metrics=["accuracy"])
cnn.summary()
t0 = time.time()
cnn_h = cnn.fit(train_ds, validation_data=val_ds, epochs=1 if QUICK else 5,
                callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)])
cnn_time = time.time() - t0
plot_history(cnn_h.history, "Custom CNN", "results/cnn_curves.png")
_, _, _, cnn_acc = evaluate_classifier(cnn, test_ds, "Custom CNN")

## A5. Main model: EfficientNetB0 (transfer learning)
Stage 1: freeze the ImageNet-pretrained base and train only the new head.
Stage 2: unfreeze the top layers and fine-tune with a small learning rate.

In [ ]:
base = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                            input_shape=(CLS_IMG, CLS_IMG, 3))
base.trainable = False
inp = layers.Input((CLS_IMG, CLS_IMG, 3))            # raw pixels 0-255, EfficientNet scales inside
x = make_augment()(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation="sigmoid")(x)
clf = tf.keras.Model(inp, out, name="crack_classifier_effnetb0")

callbacks = [tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
             tf.keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=1)]
clf.compile(tf.keras.optimizers.Adam(1e-3), "binary_crossentropy", metrics=["accuracy"])
t0 = time.time()
h1 = clf.fit(train_ds, validation_data=val_ds, epochs=1 if QUICK else 3, callbacks=callbacks)

base.trainable = True
for l in base.layers[:-30]: l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization): l.trainable = False
clf.compile(tf.keras.optimizers.Adam(1e-5), "binary_crossentropy", metrics=["accuracy"])
e0 = len(h1.history["loss"])
h2 = clf.fit(train_ds, validation_data=val_ds, epochs=e0 + (1 if QUICK else 3),
             initial_epoch=e0, callbacks=callbacks)
clf_time = time.time() - t0
clf_hist = {k: h1.history[k] + h2.history[k] for k in ["accuracy", "val_accuracy", "loss", "val_loss"]}
plot_history(clf_hist, "EfficientNetB0", "results/effnet_curves.png")

## A6. Test results (accuracy, precision, recall, F1, confusion matrix, ROC)

In [ ]:
y_true, y_prob, y_pred, clf_acc = evaluate_classifier(clf, test_ds, "EfficientNetB0")
report = classification_report(y_true, y_pred, labels=[0, 1], target_names=CLASS_NAMES, digits=4, zero_division=0)
print(report); open("results/classification_report.txt", "w").write(report)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
fpr, tpr, _ = roc_curve(y_true, y_prob) if len(set(y_true)) > 1 else ([0, 1], [0, 1], None)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(11, 4.5))
plt.subplot(1, 2, 1); plt.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks([0, 1], CLASS_NAMES); plt.yticks([0, 1], CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.subplot(1, 2, 2); plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}"); plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC Curve"); plt.legend()
plt.tight_layout(); plt.savefig("results/confusion_matrix_roc.png", dpi=110); plt.show()

plt.figure(figsize=(5, 4))
bars = plt.bar(["Custom CNN", "EfficientNetB0"], [cnn_acc, clf_acc], color=["gray", "tab:green"])
for b in bars: plt.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{b.get_height():.2%}", ha="center", va="bottom")
plt.ylim(0, 1.08); plt.ylabel("Test accuracy"); plt.title("Model comparison")
plt.tight_layout(); plt.savefig("results/model_comparison.png", dpi=110); plt.show()

# Part B: Crack Localization (Segmentation)
## B1. Download dataset
800 concrete surface images (256x256) with hand-labelled crack masks (white = crack). Split: 640 train / 80 validation / 80 test.

In [ ]:
seg_dir = snapshot_download("Vizuara/concrete-crack-dataset", repo_type="dataset", local_dir="data/seg")
imgs = sorted(glob.glob(f"{seg_dir}/images/*.jpg"))
pairs = [(p, p.replace("/images/", "/masks/").replace(".jpg", ".png")) for p in imgs]
pairs = [p for p in pairs if os.path.exists(p[1])]
idx = np.random.RandomState(SEED).permutation(len(pairs))
pairs = [pairs[i] for i in idx]
n = len(pairs); a, b = int(n * 0.8), int(n * 0.9)
seg_splits = {"train": pairs[:a], "val": pairs[a:b], "test": pairs[b:]}
print({k: len(v) for k, v in seg_splits.items()})

def read_pair(ip, mp):
    img = tf.cast(tf.image.resize(tf.io.decode_jpeg(tf.io.read_file(ip), channels=3), (SEG_IMG, SEG_IMG)), tf.float32)
    m = tf.image.resize(tf.io.decode_png(tf.io.read_file(mp), channels=1), (SEG_IMG, SEG_IMG), method="nearest")
    return img, tf.cast(m > 0, tf.float32)

def flip_pair(img, m):
    both = tf.concat([img, m], -1)
    both = tf.image.random_flip_left_right(both); both = tf.image.random_flip_up_down(both)
    both = tf.image.rot90(both, tf.random.uniform([], 0, 4, tf.int32))
    img, m = both[..., :3], both[..., 3:]
    img = tf.clip_by_value(tf.image.random_brightness(img, 25.0), 0, 255)
    return img, m

def make_seg_ds(items, training):
    ds = tf.data.Dataset.from_tensor_slices(([i for i, _ in items], [m for _, m in items]))
    ds = ds.map(read_pair, num_parallel_calls=AUTOTUNE).cache()
    if training: ds = ds.shuffle(len(items), seed=SEED).map(flip_pair, num_parallel_calls=AUTOTUNE)
    return ds.batch(16).prefetch(AUTOTUNE)

seg_train = make_seg_ds(seg_splits["train"][:64] if QUICK else seg_splits["train"], True)
seg_val   = make_seg_ds(seg_splits["val"], False)
seg_test  = make_seg_ds(seg_splits["test"], False)

xb, mb = next(iter(make_seg_ds(seg_splits["train"], False)))
plt.figure(figsize=(12, 6))
for i in range(4):
    plt.subplot(2, 4, i + 1); plt.imshow(xb[i].numpy().astype("uint8")); plt.axis("off"); plt.title("Image")
    plt.subplot(2, 4, i + 5); plt.imshow(mb[i, ..., 0], cmap="gray"); plt.axis("off"); plt.title("Crack mask")
plt.tight_layout(); plt.savefig("results/seg_samples.png", dpi=110); plt.show()

## B2. U-Net model (MobileNetV2 encoder)
The encoder (pretrained MobileNetV2) understands the image, the decoder rebuilds a full-size crack map. Skip connections keep thin crack details.
Loss = Binary cross-entropy + Dice loss (good for thin objects like cracks).

In [ ]:
def dice_coef(y, p, eps=1.0):
    y, p = tf.reshape(y, [-1]), tf.reshape(p, [-1])
    return (2 * tf.reduce_sum(y * p) + eps) / (tf.reduce_sum(y) + tf.reduce_sum(p) + eps)

def bce_dice_loss(y, p):
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y, p)) + (1 - dice_coef(y, p))

def dice_metric(y, p): return dice_coef(y, tf.cast(p > 0.5, tf.float32))
def iou_metric(y, p):
    p = tf.cast(p > 0.5, tf.float32)
    inter = tf.reduce_sum(y * p); union = tf.reduce_sum(y) + tf.reduce_sum(p) - inter
    return (inter + 1.0) / (union + 1.0)

def build_unet():
    inp = layers.Input((SEG_IMG, SEG_IMG, 3))          # raw pixels 0-255
    x = layers.Rescaling(1 / 127.5, offset=-1)(inp)
    enc = tf.keras.applications.MobileNetV2(input_shape=(SEG_IMG, SEG_IMG, 3), include_top=False, weights="imagenet")
    names = ["block_1_expand_relu", "block_3_expand_relu", "block_6_expand_relu", "block_13_expand_relu", "block_16_project"]
    encoder = tf.keras.Model(enc.input, [enc.get_layer(n).output for n in names], name="mobilenetv2_encoder")
    skips = encoder(x)
    x, skips = skips[-1], reversed(skips[:-1])
    for f, s in zip([256, 128, 64, 32], skips):
        x = layers.Conv2DTranspose(f, 3, strides=2, padding="same")(x)
        x = layers.Concatenate()([x, s])
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu")(x)
    out = layers.Conv2D(1, 1, activation="sigmoid")(x)
    return tf.keras.Model(inp, out, name="crack_unet")

unet = build_unet()
unet.compile(tf.keras.optimizers.Adam(3e-4), loss=bce_dice_loss, metrics=[dice_metric, iou_metric])
print("U-Net parameters:", f"{unet.count_params():,}")
t0 = time.time()
seg_h = unet.fit(seg_train, validation_data=seg_val, epochs=1 if QUICK else 40, callbacks=[
    tf.keras.callbacks.EarlyStopping(monitor="val_dice_metric", mode="max", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_dice_metric", mode="max", factor=0.5, patience=4)])
seg_time = time.time() - t0
plot_history(seg_h.history, "U-Net", "results/unet_curves.png", metric="dice_metric")

## B3. Segmentation test results (Dice and IoU)

In [ ]:
dices, ious = [], []
for x, m in seg_test:
    p = unet.predict(x, verbose=0)
    for i in range(len(x)):
        if m[i].numpy().sum() > 0:                        # score only images that really have a crack
            dices.append(float(dice_metric(m[i], p[i]))); ious.append(float(iou_metric(m[i], p[i])))
seg_dice, seg_iou = float(np.mean(dices)), float(np.mean(ious))
print(f"Test Dice: {seg_dice:.4f} | Test IoU: {seg_iou:.4f}  (on {len(dices)} cracked test images)")

xb, mb = next(iter(seg_test)); pb = unet.predict(xb, verbose=0)
plt.figure(figsize=(12, 12))
for r in range(4):
    img = xb[r].numpy().astype("uint8"); over = img.copy(); over[pb[r, ..., 0] > 0.5] = [255, 0, 0]
    for c, (im, t) in enumerate([(img, "Image"), (mb[r, ..., 0], "True mask"), (pb[r, ..., 0] > 0.5, "Predicted mask"), (over, "Overlay")]):
        plt.subplot(4, 4, r * 4 + c + 1); plt.imshow(im, cmap="gray" if c in (1, 2) else None); plt.axis("off"); plt.title(t)
plt.tight_layout(); plt.savefig("results/segmentation_results.png", dpi=110); plt.show()

# Part C: Crack Inspector (full system)
Combines both models. Large photos are split into 256x256 tiles so thin cracks are not lost.
Severity is a rough guide from crack area: under 1% = Minor, 1 to 3% = Moderate, over 3% = Severe.

In [ ]:
def segment_full(img):
    """img: HxWx3 uint8. Returns HxW crack probability map."""
    h, w = img.shape[:2]
    if max(h, w) <= 384:
        p = unet.predict(tf.image.resize(img[None].astype("float32"), (SEG_IMG, SEG_IMG)), verbose=0)[0]
        return tf.image.resize(p, (h, w)).numpy()[..., 0]
    T = SEG_IMG; H, W = int(np.ceil(h / T)) * T, int(np.ceil(w / T)) * T
    pad = np.pad(img, ((0, H - h), (0, W - w), (0, 0)), mode="reflect").astype("float32")
    tiles = pad.reshape(H // T, T, W // T, T, 3).transpose(0, 2, 1, 3, 4).reshape(-1, T, T, 3)
    p = unet.predict(tiles, batch_size=16, verbose=0)[..., 0]
    return p.reshape(H // T, W // T, T, T).transpose(0, 2, 1, 3).reshape(H, W)[:h, :w]

def inspect(img):
    img = np.array(img.convert("RGB") if hasattr(img, "convert") else img).astype("uint8")
    prob = float(clf.predict(tf.image.resize(img[None].astype("float32"), (CLS_IMG, CLS_IMG)), verbose=0)[0, 0])
    mask = segment_full(img) > 0.5
    area = float(mask.mean() * 100)
    crack = prob > 0.5 or area > 0.5
    severity = "None" if not crack else ("Minor" if area < 1 else "Moderate" if area < 3 else "Severe")
    over = img.copy(); over[mask] = (0.35 * over[mask] + 0.65 * np.array([255, 0, 0])).astype("uint8")
    return {"crack": crack, "confidence": prob if crack else 1 - prob, "area_pct": area, "severity": severity}, over

test_imgs = [p for p, _ in seg_splits["test"][:3]] + glob.glob("data/cls/test/Negative/*")[:1] + glob.glob("data/cls/test/Positive/*")[:2]
from PIL import Image
plt.figure(figsize=(15, 3.8))
for i, f in enumerate(test_imgs):
    r, over = inspect(Image.open(f))
    plt.subplot(1, 6, i + 1); plt.imshow(over); plt.axis("off")
    plt.title(f"{'CRACK' if r['crack'] else 'NO CRACK'} ({r['confidence']:.0%})\nArea {r['area_pct']:.2f}% | {r['severity']}",
              fontsize=9, color="red" if r["crack"] else "green")
plt.suptitle("Crack Inspector output"); plt.tight_layout(); plt.savefig("results/inspector_demo.png", dpi=110); plt.show()

## C2. Try your own photo

In [ ]:
try:
    from google.colab import files
    for name in files.upload():
        r, over = inspect(Image.open(name))
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1); plt.imshow(Image.open(name)); plt.axis("off"); plt.title("Input")
        plt.subplot(1, 2, 2); plt.imshow(over); plt.axis("off")
        plt.title(f"{'CRACK' if r['crack'] else 'NO CRACK'} | Area {r['area_pct']:.2f}% | {r['severity']}")
        plt.show(); print(r)
except ImportError:
    print("Upload works in Google Colab only. Locally use: python src/predict.py photo.jpg")

# Save models, metrics and download

In [ ]:
def save_light(m, path):          # save without optimizer state so files stay small for GitHub
    tf.keras.Model(m.inputs, m.outputs, name=m.name).save(path)
save_light(clf, "models/crack_classifier.keras")
save_light(unet, "models/crack_unet.keras")
metrics = {
    "classification": {
        "dataset": "Concrete Crack Images for Classification (40,000 images)",
        "custom_cnn": {"test_accuracy": round(cnn_acc, 4), "params": int(cnn.count_params()), "train_min": round(cnn_time / 60, 1)},
        "efficientnet_b0": {"test_accuracy": round(clf_acc, 4), "roc_auc": round(float(roc_auc), 4),
                            "params": int(clf.count_params()), "train_min": round(clf_time / 60, 1)}},
    "segmentation": {
        "dataset": "Concrete crack segmentation (800 images + masks)",
        "unet_mobilenetv2": {"test_dice": round(seg_dice, 4), "test_iou": round(seg_iou, 4),
                             "params": int(unet.count_params()), "train_min": round(seg_time / 60, 1)}},
}
json.dump(metrics, open("results/metrics.json", "w"), indent=2)
print(json.dumps(metrics, indent=2))
for f in glob.glob("models/*.keras"):
    mb = os.path.getsize(f) / 1e6
    print(f, "%.1f MB" % mb, "(OK for GitHub upload)" if mb < 25 else "(too big for browser upload, see README)")

In [ ]:
!zip -qr trained_outputs.zip models results
try:
    from google.colab import files; files.download("trained_outputs.zip")
except ImportError:
    pass

# (Optional) Live demo web app for viva
Creates a public link for 72 hours. Open it, upload a photo, see the result.

In [ ]:
!pip install -q gradio
import gradio as gr
def demo_fn(image):
    r, over = inspect(Image.fromarray(image))
    txt = (f"Result: {'CRACK DETECTED' if r['crack'] else 'NO CRACK'}\n"
           f"Confidence: {r['confidence']:.1%}\nCrack area: {r['area_pct']:.2f}%\nSeverity: {r['severity']}")
    return over, txt
gr.Interface(demo_fn, gr.Image(label="Wall / concrete photo"),
             [gr.Image(label="Crack location (red)"), gr.Textbox(label="Report")],
             title="Building Crack Inspector").launch(share=True)